# Story 1.1: Agent Integration Test

Simple test to verify:
1. Scenario Generator Agent creates scenarios
2. Simulation Feedback Agent provides feedback
3. MLflow traces are logged to summit-sim experiment

## 1. Setup & Imports

In [1]:
import mlflow

from summit_sim.agents.generator import generate_scenario
from summit_sim.agents.simulation import process_choice
from summit_sim.schemas import HostConfig
from summit_sim.settings import settings

# Initialize MLflow
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
mlflow.pydantic_ai.autolog()

print(f"✓ MLflow tracking URI: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment name: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key configured: {bool(settings.openrouter_api_key)}")

✓ MLflow tracking URI: https://mlflow.bhamm-lab.com
✓ Experiment name: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key configured: True


## 2. Test Scenario Generator

In [6]:
# Test configuration
config = HostConfig(num_participants=3, activity_type="hiking", difficulty="med")

print(f"Generating scenario: {config}")
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Starting turn: {scenario.starting_turn_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating scenario: num_participants=3 activity_type='hiking' difficulty='med'
------------------------------------------------------------
✓ Title: High Ridge Fracture
✓ Setting: High Ridge Trail, 5 miles from the trailhead in a mountainous region. It is 3:30 PM, the temperature is 50°F (10°C) and dropping rapidly. The area is isolated with poor cell reception.
✓ Patient: 35-year-old male, experienced hiker. Slipped on loose scree while descending a steep section. Conscious, alert, but in severe pain. His left knee is visibly deformed and swollen.
✓ Turns: 3
✓ Starting turn: initial_assessment

Learning Objectives:
  • Perform a systematic patient assessment in a wilderness setting.
  • Apply appropriate immobilization techniques for lower-extremity fractures.
  • Prioritize environmental protection and triage for evacuation.


Trace(trace_id=tr-042dc390f54f82e6e5d35922a263e4e1)

## 3. Test Simulation Feedback (Single Turn)

In [3]:
# Get starting turn
current_turn = scenario.get_turn(scenario.starting_turn_id)

print(f"Turn {current_turn.turn_id}:")
print(f"{current_turn.narrative_text}")
print("\nChoices:")
for i, choice in enumerate(current_turn.choices):
    print(f"  {i + 1}. {choice.description}")

# Select first choice
selected_choice = current_turn.choices[0]
print(f"\n>>> Selecting: {selected_choice.description}")
print("-" * 60)

# Get feedback
result = await process_choice(scenario, current_turn, selected_choice)

print("\nFeedback:")
print(result.feedback)
print("\nLearning Moments:")
for moment in result.learning_moments:
    print(f"  • {moment}")
print(f"\n✓ Scenario complete: {result.is_complete}")

Turn initial_assessment:
You are hiking on an exposed ridge with two companions, Mike and Sarah. It's late afternoon and the weather is cooling rapidly. Sarah suddenly stops, says she feels 'dizzy, nauseous, and super weird,' and sits down heavily on a rock. She looks pale; you notice her hands are shaking.

Choices:
  1. Immediately conduct a thorough primary assessment, checking vitals, mental status, and performing a quick head-to-toe check.
  2. Encourage Sarah to drink water, rest for 5 minutes, and then try to continue hiking at a slow pace.

>>> Selecting: Immediately conduct a thorough primary assessment, checking vitals, mental status, and performing a quick head-to-toe check.
------------------------------------------------------------

Feedback:
Excellent decision. Performing a systematic assessment is crucial, as the wilderness environment often creates 'red herrings'—like altitude—that can mask the true cause of an illness. By choosing to gather objective vitals first, you

Trace(trace_id=tr-d0a21a781783b3431ac428aafb6e3c50)

## 4. Test Multiple Activities (Optional)

In [4]:
# Quick test of all activities
activities = ["canyoneering", "skiing", "hiking"]

for activity in activities:
    config = HostConfig(num_participants=2, activity_type=activity, difficulty="low")
    test_scenario = await generate_scenario(config)
    print(f"✓ {activity}: {test_scenario.title[:50]}...")

✓ canyoneering: Slot Canyon Slip...
✓ skiing: Powder Panic: The Green Run Assessment...
✓ hiking: The Unsteady Trail...


[Trace(trace_id=tr-659afb19ba683fcae95bd3678c4c97b3), Trace(trace_id=tr-f4e65b3b7d0fd5f85d3a95f8f39463fe), Trace(trace_id=tr-05ce6754ac80567a8c8fa010cf9a2fac)]

## 5. MLflow Verification

Check your MLflow UI at the tracking URI to verify:
- Traces appear under the `summit-sim` experiment
- Each agent call is logged with latency metrics
- Token usage is tracked

In [5]:
# Show experiment info
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"\nView traces at: {settings.mlflow_tracking_uri}")

Experiment ID: 7
Artifact Location: mlflow-artifacts:/7

View traces at: https://mlflow.bhamm-lab.com


---
## ✅ Integration Test Complete

If you see checkmarks and no errors, Story 1.1 is working!